In [5]:
import xarray as xr
import torch
import yaml
import sys
from pathlib import Path
root_dir = Path.cwd().parent   
sys.path.append(str(root_dir))

from data.dataset import *
from data.dataloader import *
from data.preprocessing import *

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version (runtime):", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

Torch version: 2.3.1
CUDA available: True
CUDA version (runtime): 12.1
GPU: Quadro T1000 with Max-Q Design


In [17]:
# Load config
config_path = Path.cwd().parent / "utils" / "default_config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

data_cfg = config["data"]
train_cfg = config["training"]
print("Config loaded!")

# Load full dataset
url = data_cfg["dataset_url"]
ds = load_full_dataset(url)

# Reduce dataset using config
reduced_ds = reduce_dataset(ds, data_cfg)
reduced_ds = reduced_ds.chunk({"time": 365, "latitude": 50, "longitude": 50})
print(reduced_ds)

Config loaded!
Opening dataset from: gs://weatherbench2/datasets/era5_daily/1959-2023_01_10-full_37-1h-0p25deg-chunk-1-s2s.zarr
Full dataset loaded!
Reducing dataset...
Reduced dataset ready!
<xarray.Dataset> Size: 9GB
Dimensions:                   (time: 4748, latitude: 141, longitude: 161,
                               level: 3)
Coordinates:
  * latitude                  (latitude) float32 564B 70.0 69.75 ... 35.25 35.0
  * level                     (level) int64 24B 200 500 850
  * longitude                 (longitude) float32 644B 0.0 0.25 ... 39.75 40.0
  * time                      (time) datetime64[ns] 38kB 2010-01-01 ... 2022-...
Data variables: (12/13)
    2m_temperature            (time, latitude, longitude) float32 431MB dask.array<chunksize=(365, 50, 50), meta=np.ndarray>
    10m_u_component_of_wind   (time, latitude, longitude) float32 431MB dask.array<chunksize=(365, 50, 50), meta=np.ndarray>
    10m_v_component_of_wind   (time, latitude, longitude) float32 431MB dask.ar

In [18]:
# Build PyTorch Datasets

variables_to_keep = data_cfg["variables_to_keep"]
target_var = data_cfg["target_variable"]

train_ds = ERA5Dataset(
    reduced_ds, variables_to_keep, target_var,
    input_length=train_cfg["input_length"],
    forecast_horizon=train_cfg["forecast_horizon"],
    split="train"
)
val_ds = ERA5Dataset(
    reduced_ds, variables_to_keep, target_var,
    input_length=train_cfg["input_length"],
    forecast_horizon=train_cfg["forecast_horizon"],
    split="val"
)
test_ds = ERA5Dataset(
    reduced_ds, variables_to_keep, target_var,
    input_length=train_cfg["input_length"],
    forecast_horizon=train_cfg["forecast_horizon"],
    split="test"
)
print("\nDataset sizes:")
print("Train:", len(train_ds), " Val:", len(val_ds), " Test:", len(test_ds))


Dataset sizes:
Train: 2914  Val: 722  Test: 1088


In [ ]:
# DataLoaders
train_loader, val_loader, test_loader = get_dataloaders(
    reduced_ds, variables_to_keep, target_var,
    input_length=train_cfg["input_length"],
    forecast_horizon=train_cfg["forecast_horizon"],
    batch_size=train_cfg["batch_size"],
    num_workers=train_cfg["num_workers"]
)

# Fetch one batch
X, y = next(iter(train_loader))
print("\nExample batch:")
print("X:", X.shape, " y:", y.shape)
